# OpenShorts on Kaggle (2×T4)

**Before running — notebook settings (right panel):**

| setting | value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** (needs a phone-verified account) |

**Add-ons → Secrets** — create these and tick them for this notebook:

| secret | needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (a PAT with `repo` scope) |
| `GEMINI_API_KEY` | clip selection + scene context — **without it nothing is clipped** |
| `ASSEMBLYAI_API_KEY` | transcription *with diarization*. Without it, faster-whisper runs instead and has no diarization — which the framing policy uses, so quality drops |
| `YOUTUBE_COOKIES` | paste the whole contents of a working `cookies.txt` |

Run the cells in order. Cell 4 tells you whether it actually works.

## 1 — Is the GPU actually there?

If this shows 0 GPUs, fix the accelerator setting before going further — everything below will still run, just ~25× slower.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  devices={torch.cuda.device_count()}")

## 2 — Keys

**Two ways. Pick one.**

**A. Paste below (quickest).** Fill the strings in the next cell.

> ⚠️ Anything typed into a cell is saved *inside the notebook*, including in
> Kaggle's version history. That is fine for a private notebook you never
> share — but if you ever make it public, fork it to a teammate, or download
> and commit it, **the keys go with it**. There is no way to un-leak a key
> from a saved version; you have to rotate it.

**B. Kaggle Secrets (safer, survives sharing).** Leave the strings empty and
create them under **Add-ons → Secrets** with these exact labels:
`GITHUB_TOKEN`, `GEMINI_API_KEY`, `ASSEMBLYAI_API_KEY`, `YOUTUBE_COOKIES`.
Secrets must be created in Kaggle's UI — a notebook can only read them.

The cell below tries the pasted value first, then falls back to Secrets, so
either way works and you can mix them (e.g. paste the token, keep cookies in
a secret).

In [ ]:
# ─────────── PASTE HERE (or leave blank to use Kaggle Secrets) ───────────
GITHUB_TOKEN       = ""   # PAT with Contents:Read — only if the repo is private
GEMINI_API_KEY     = ""   # REQUIRED: no clips are selected without it
# Extra Gemini keys, comma-separated. gemini_pool.GeminiKeyPool rotates to
# the next one on 429/quota/503, which is what keeps a long job alive on
# free-tier keys. Same list as the Settings "extra keys" box in the UI.
GEMINI_API_KEYS    = ""   # e.g. "key2,key3,key4"
ASSEMBLYAI_API_KEY = ""   # optional: enables diarization (better framing)
# Optional. Stage 3 clip selection runs the viral-clip-finder engine on
# GEMINI_API_KEY above. This SEPARATE key is what the older narrative
# engine uses as the automatic fallback; without it there is no second
# Stage 3 path, and it also keeps Stage 3 off the vision key's quota.
NARRATIVE_GEMINI_API_KEY = ""

# Cookies are multi-line, so use the triple quotes. Paste the whole
# cookies.txt including the '# Netscape HTTP Cookie File' header line.
YOUTUBE_COOKIES = """
"""
# ─────────────────────────────────────────────────────────────────────────

import os

_pasted = {"GITHUB_TOKEN": GITHUB_TOKEN, "GEMINI_API_KEY": GEMINI_API_KEY,
           "GEMINI_API_KEYS": GEMINI_API_KEYS,
           "ASSEMBLYAI_API_KEY": ASSEMBLYAI_API_KEY,
           "NARRATIVE_GEMINI_API_KEY": NARRATIVE_GEMINI_API_KEY,
           "YOUTUBE_COOKIES": YOUTUBE_COOKIES}

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None

def load(name, required=False):
    """Pasted value wins; otherwise fall back to a Kaggle Secret."""
    val = (_pasted.get(name) or "").strip()
    src = "pasted"
    if not val and _secrets is not None:
        try:
            val, src = _secrets.get_secret(name).strip(), "secret"
        except Exception:
            val = ""
    if val:
        os.environ[name] = val
        print(f"  ok       {name:<20} ({src}, {len(val)} chars)")
        return True
    print(f"  {'MISSING ' if required else 'not set '} {name}")
    return False

load("GEMINI_API_KEY", required=True)
if load("GEMINI_API_KEYS"):
    n = len([k for k in os.environ["GEMINI_API_KEYS"].split(",") if k.strip()])
    print(f"           -> key pool: 1 primary + {n} extra")
load("ASSEMBLYAI_API_KEY")
if not load("NARRATIVE_GEMINI_API_KEY"):
    print("           -> Stage 3 runs the skill engine on GEMINI_API_KEY;\n              the narrative fallback stays inert")
load("YOUTUBE_COOKIES")
have_token = load("GITHUB_TOKEN")

In [ ]:
import os, subprocess

BRANCH = "session/framing-work"
DEST = "/kaggle/working/openshorts"
PRIVATE_REPO = True   # set False if you make the repo public

# Fail loudly and specifically. The common cause of a clone failure here is
# NOT a bad token — it is a secret that was saved but never ATTACHED to this
# notebook. Kaggle requires ticking the checkbox next to each secret in
# Add-ons -> Secrets; until then get_secret() raises and this cell would
# otherwise fall through to an anonymous clone, which a private repo answers
# with 'could not read Username for https://github.com'.
if PRIVATE_REPO and not have_token:
    raise SystemExit(
        "GITHUB_TOKEN not loaded.\n"
        "  -> Add-ons > Secrets: TICK THE CHECKBOX next to GITHUB_TOKEN\n"
        "     (saving the secret is not enough; it must be attached to this\n"
        "     notebook), then re-run the keys cell above and this one.")

url = (f"https://{os.environ['GITHUB_TOKEN']}@github.com/foskigr8/openshorts.git"
       if have_token else "https://github.com/foskigr8/openshorts.git")

if not os.path.isdir(DEST):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # Never print the URL — it carries the token.
        err = r.stderr[-400:]
        hint = ""
        if "could not read Username" in err:
            hint = "\n  -> the token was not applied; see the checkbox note above"
        elif "Authentication failed" in err or "403" in err:
            hint = ("\n  -> token rejected: check it has Contents:Read on"
                    " foskigr8/openshorts and has not expired")
        elif "Remote branch" in err:
            hint = f"\n  -> branch '{BRANCH}' not found on the remote"
        raise SystemExit(f"CLONE FAILED:\n{err}{hint}")
    print("clone ok")
else:
    print("already cloned")

os.chdir(DEST)
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


## 3 — Install and launch

First run takes **5–10 min** (pip + npm build). It ends by printing a public URL.

In [ ]:
!bash kaggle_bootstrap.sh

## 4 — Smoke test: is it *actually* working?

Starting is not the same as working. This checks the four things that
independently break, and says which one failed rather than just "error".

In [ ]:
import json, os, subprocess, urllib.request

DEST = globals().get("DEST", "/kaggle/working/openshorts")

def check(label, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        ok, detail = False, f"{type(e).__name__}: {e}"
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}: {detail}")
    return ok

def api(path):
    with urllib.request.urlopen(f"http://localhost:8000{path}", timeout=15) as r:
        return r.status, r.read()

results = []

# 1. API alive
results.append(check("API", lambda: (api("/api/system")[0] == 200, "/api/system 200")))

# 2. Dashboard served from the SAME process (single-origin mode)
def _ui():
    status, body = api("/")
    return status == 200 and b"<div id=\"root\"" in body, f"/ returned {len(body)} bytes of HTML"
results.append(check("Dashboard", _ui))

# 3. GPU reachable from the app's own imports (not just nvidia-smi)
def _gpu():
    out = subprocess.run(["python3", "-c",
        "import torch;print(torch.cuda.is_available(), torch.cuda.device_count())"],
        capture_output=True, text=True).stdout.strip()
    return out.startswith("True"), out
results.append(check("CUDA in app env", _gpu))

# 4. NVENC present — without it every render silently falls back to x264 (slow)
def _nvenc():
    out = subprocess.run("ffmpeg -hide_banner -encoders 2>/dev/null | grep -c nvenc",
                         shell=True, capture_output=True, text=True).stdout.strip()
    return out.isdigit() and int(out) > 0, f"{out} nvenc encoders"
results.append(check("NVENC", _nvenc))

# 5. YouTube reachable with the cookie jar — FUNCTIONAL, not structural.
#    cookie_health.py only checks that cookie NAMES exist and will report OK
#    for a jar YouTube rejects, so it is deliberately not used here.
def _yt():
    r = subprocess.run(["yt-dlp", "--cookies", "cookies.txt", "--skip-download",
                        "--print", "%(title)s", "https://youtu.be/ua9Z0Lq3QVA"],
                       capture_output=True, text=True, timeout=120)
    title = (r.stdout or "").strip().splitlines()[-1] if r.stdout.strip() else ""
    return bool(title), title or (r.stderr or "").strip()[-200:]
results.append(check("YouTube + cookies", _yt))

# 6. Stage 3 engine — the clip SELECTOR, which decides what gets rendered.
#    A missing skill package or provider key does not crash anything: the job
#    silently drops to the older narrative engine (or, with neither key, to the
#    legacy window scorer) and produces weaker clips that still look like a
#    successful run. Check it while it is cheap to fix.
def _stage3():
    out = subprocess.run(["python3", "-c", (
        "import os, viral_clip_finder as v;"
        "print(v.skill_available(), len(v._provider_candidates()),"
        " os.environ.get('VIRAL_ENGINE', 'auto'))")],
        capture_output=True, text=True, cwd=DEST)
    parts = (out.stdout or "").strip().split()
    if len(parts) != 3:
        return False, (out.stderr or out.stdout).strip()[-200:]
    have_skill, providers, engine = parts[0] == "True", int(parts[1]), parts[2]
    if not have_skill:
        return False, "viral_clip_finder_skill/ is missing from the clone"
    if providers == 0:
        return False, "no Stage 3 provider key (GEMINI_API_KEY / NARRATIVE_GEMINI_API_KEY)"
    return True, f"engine={engine}, skill package present, {providers} provider(s)"
results.append(check("Stage 3 clip selection", _stage3))

print(f"\n{sum(results)}/{len(results)} passed")
if not all(results):
    print("\nIf 'YouTube + cookies' failed: the jar expired (~3h lifetime).")
    print("Re-paste YOUTUBE_COOKIES from a fresh local cookies.txt and re-run cell 2-3.")

## 5 — End-to-end render test (optional, ~1-2 min on GPU)

The real proof: reframe a short clip and check the framing telemetry.
Uses a source you upload or download yourself, so it works even if cookies
are dead.

In [ ]:
# Point SRC at any short landscape video with speech (upload one to
# /kaggle/working, or attach a Kaggle Dataset and use /kaggle/input/...).
SRC = "/kaggle/working/test.mp4"

import os, time
if not os.path.exists(SRC):
    print(f"put a video at {SRC} first (or edit SRC)")
else:
    os.environ.setdefault("USE_ASD", "1")
    import reframe_v2 as r
    t0 = time.time()
    # No transcript here, so diarization is unavailable and LR-ASD carries the
    # speaker identification alone — a deliberately harder case than production.
    r.render(SRC, "/kaggle/working/test_vertical.mp4", 0.75)
    print(f"\nrendered in {time.time()-t0:.1f}s -> /kaggle/working/test_vertical.mp4")
    print("Look at the '🎯 Framing evidence' line above:")
    print("  lip-sync high + size ~0%  = working as intended")
    print("  size high                 = the speaker signal is not reaching the camera")

## 6 — Keep the session alive (non-blocking)

Jupyter runs **one cell at a time**, so a blocking `while True:` loop locks
the notebook — you cannot run anything else without interrupting it. This
runs the heartbeat on a background thread instead, so the cell returns
immediately and you keep using the notebook.

You do not need this while you are actively clicking around: Kaggle keeps
an interactive session alive on its own. It matters when you walk away
mid-render, or use *Save & Run All*.

**Stopping any cell is safe.** The backend and tunnel are started with
`nohup` as background processes, so interrupting a notebook cell never
kills them.

**Outputs do not survive the session.** `/kaggle/working` is wiped at the
end (12h cap). Download clips through the dashboard before stopping, or set
the `AWS_*` secrets so `s3_uploader.py` pushes them out as they finish.

In [ ]:
import threading, time

def _heartbeat():
    # Touches a file rather than printing: notebook output from a background
    # thread interleaves with whatever cell you are running next, which makes
    # the notebook unreadable.
    while True:
        with open('/kaggle/working/.heartbeat', 'w') as f:
            f.write(str(time.time()))
        time.sleep(60)

if not any(t.name == 'openshorts-heartbeat' for t in threading.enumerate()):
    threading.Thread(target=_heartbeat, name='openshorts-heartbeat',
                     daemon=True).start()
    print('heartbeat started on a background thread — this cell is free')
else:
    print('heartbeat already running')
